This file is for testing gradient accumulation method

also i will combine this with mixed precision training

In [1]:
import pandas as pd
import regex as re
import torch

In [2]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [3]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [4]:
from pathlib import Path

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
from torch.utils.data import Dataset
from PIL import Image

In [9]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
image_paths = list(path.rglob("*.png"))

In [11]:
from torchvision import transforms
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [12]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [13]:
import kornia.augmentation as K
import torch.nn as nn

In [14]:
transform_1 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_1 = dset(image_paths = image_paths, transform = transform_1)

ts = int(0.75*len(dataset_1))
vs = len(dataset_1) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_1, [ts, vs], generator)

train_loader_1 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_1 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_1 = torchvision.models.resnet34(weights="DEFAULT")
model_1.fc = nn.Linear(
    model_1.fc.in_features,
    num_classes
)

model_1 = model_1.to(device)

for param in model_1.parameters():
    param.requires_grad = False

for param in model_1.fc.parameters():
    param.requires_grad = True

train_aug_1 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_1 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_1 = torch.optim.RMSprop(
    model_1.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_1)
)

criterion = nn.CrossEntropyLoss()

accum_steps = 4

for epoch in range(20):
    model_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    optimizer_1.zero_grad()

    for step, (images, labels) in enumerate(train_loader_1):

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_1(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_1(images[mask])

        outputs = model_1(images)

        loss = criterion(outputs, labels) / accum_steps

        loss.backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader_1):
            optimizer_1.step()
            optimizer_1.zero_grad()
            scheduler_1.step()

        train_loss += loss.item() * accum_steps

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_1:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 49.92% | Valid Acc: 81.50%
Epoch 2/20 | Train Acc: 85.09% | Valid Acc: 88.15%
Epoch 3/20 | Train Acc: 91.04% | Valid Acc: 91.43%
Epoch 4/20 | Train Acc: 94.44% | Valid Acc: 93.55%
Epoch 5/20 | Train Acc: 95.70% | Valid Acc: 94.99%
Epoch 6/20 | Train Acc: 96.95% | Valid Acc: 96.82%
Epoch 7/20 | Train Acc: 97.01% | Valid Acc: 95.18%
Epoch 8/20 | Train Acc: 96.56% | Valid Acc: 96.34%
Epoch 9/20 | Train Acc: 95.79% | Valid Acc: 95.66%
Epoch 10/20 | Train Acc: 95.92% | Valid Acc: 96.34%
Epoch 11/20 | Train Acc: 96.43% | Valid Acc: 95.38%
Epoch 12/20 | Train Acc: 95.31% | Valid Acc: 95.38%
Epoch 13/20 | Train Acc: 92.61% | Valid Acc: 93.35%
Epoch 14/20 | Train Acc: 95.41% | Valid Acc: 96.72%
Epoch 15/20 | Train Acc: 96.24% | Valid Acc: 97.59%
Epoch 16/20 | Train Acc: 95.66% | Valid Acc: 94.99%
Epoch 17/20 | Train Acc: 95.25% | Valid Acc: 94.80%
Epoch 18/20 | Train Acc: 92.58% | Valid Acc: 94.89%
Epoch 19/20 | Train Acc: 94.70% | Valid Acc: 96.24%
Epoch 20/20 | Train A

Now mixing gradient acc and mixed prec

In [15]:
transform_2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_2 = dset(image_paths=image_paths, transform=transform_2)

ts = int(0.75*len(dataset_2))
vs = len(dataset_2) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_2, [ts, vs], generator)

train_loader_2 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_2 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_2 = torchvision.models.resnet34(weights="DEFAULT")
model_2.fc = nn.Linear(
    model_2.fc.in_features,
    num_classes
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = False

for param in model_2.fc.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast=0.2,
        p=0.5
    ),
    K.RandomPlanckianJitter(
        mode="CIED",
        p=0.5
    )
).to(device)

targeted_aug_2 = K.AugmentationSequential(
    K.RandomRotation(
        degrees=10,
        p=0.5
    ),
    K.RandomAffine(
        degrees=0,
        scale=(0.9, 1.1),
        p=0.5
    ),
    K.RandomPerspective(
        distortion_scale=0.2,
        p=0.5
    ),
    K.ColorJiggle(
        brightness=0.2,
        contrast=0.2,
        p=0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr=lr_head,
    epochs=20,
    steps_per_epoch=len(train_loader_2)
)

criterion = nn.CrossEntropyLoss()

accum_steps = 4
scaler_2 = torch.cuda.amp.GradScaler()

for epoch in range(20):
    model_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    optimizer_2.zero_grad()

    for step, (images, labels) in enumerate(train_loader_2):

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_2(images[mask])

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model_2(images)
            loss = criterion(outputs, labels) / accum_steps

        scaler_2.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader_2):
            scaler_2.step(optimizer_2)
            scaler_2.update()
            optimizer_2.zero_grad()
            scheduler_2.step()

        train_loss += loss.item() * accum_steps

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_2:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

C:\Users\Nilansh Barotia\AppData\Local\Temp\ipykernel_30128\962514416.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_2 = torch.cuda.amp.GradScaler()


Epoch 1/20 | Train Acc: 44.84% | Valid Acc: 77.55%
Epoch 2/20 | Train Acc: 83.78% | Valid Acc: 90.27%
Epoch 3/20 | Train Acc: 91.84% | Valid Acc: 92.00%
Epoch 4/20 | Train Acc: 94.89% | Valid Acc: 94.51%
Epoch 5/20 | Train Acc: 96.08% | Valid Acc: 94.89%
Epoch 6/20 | Train Acc: 95.76% | Valid Acc: 95.66%
Epoch 7/20 | Train Acc: 96.27% | Valid Acc: 95.66%
Epoch 8/20 | Train Acc: 95.92% | Valid Acc: 96.15%
Epoch 9/20 | Train Acc: 96.79% | Valid Acc: 96.72%
Epoch 10/20 | Train Acc: 96.40% | Valid Acc: 96.82%
Epoch 11/20 | Train Acc: 95.89% | Valid Acc: 95.66%
Epoch 12/20 | Train Acc: 96.21% | Valid Acc: 95.86%
Epoch 13/20 | Train Acc: 94.89% | Valid Acc: 96.92%
Epoch 14/20 | Train Acc: 95.28% | Valid Acc: 96.53%
Epoch 15/20 | Train Acc: 95.34% | Valid Acc: 95.86%
Epoch 16/20 | Train Acc: 93.99% | Valid Acc: 94.22%
Epoch 17/20 | Train Acc: 95.47% | Valid Acc: 95.57%
Epoch 18/20 | Train Acc: 94.25% | Valid Acc: 97.11%
Epoch 19/20 | Train Acc: 94.67% | Valid Acc: 93.93%
Epoch 20/20 | Train A